# MP03 — Image Resizing Laboratory
Implementation experiments executed; learner mastery remains unassessed. Objective: understand nearest-neighbor, bilinear and bicubic interpolation.

## Intuition and coordinates
An image samples a spatial signal on a grid. Resizing estimates values on another grid; extra output pixels are not extra measurements. Our arrays are H×W×C: H rows, W columns, C bands. Values are floating point in explicitly declared units.

For an output row j, map to input coordinate u=(j+1/2)H/Hout−1/2. Hout is the output height; j starts at zero. Columns use the same formula with W. This half-pixel convention aligns pixel-cell boundaries. Clamp coordinates outside the source to its nearest edge. Other conventions can yield different numbers, especially near boundaries.

Nearest neighbor takes floor(u+1/2), choosing the larger index at ties. Bilinear interpolation uses four neighboring pixels. With lower row i=floor(u), lower column k=floor(v), a=u−i and b=v−k, the weights are (1−a)(1−b), (1−a)b, a(1−b), ab. They sum to one and are nonnegative. C bands are interpolated independently. Bilinear output therefore stays within the contributing range.

For a one-row image [0,10] resized to width four, source coordinates clamp to [0,0.25,0.75,1]. Bilinear predicts [0,2.5,7.5,10]; nearest predicts [0,0,10,10]. Predict these values before running.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp03_tools import resize_hwc,compare_hwc
x=np.array([[[0.],[10.]]])
for method in ('nearest','bilinear'): print(method,resize_hwc(x,(1,4),method).ravel())
np.testing.assert_allclose(resize_hwc(x,(1,4)).ravel(),[0,2.5,7.5,10])

## Bicubic and aliasing
Bicubic combines a 4×4 neighborhood with cubic weights. Negative weights can produce overshoot near sharp edges; clipping before measuring error would change the experiment. This lesson uses the declared PyTorch bicubic implementation rather than claiming a new bicubic implementation. Its settings are align_corners=False and antialias=False.

Downsampling can make high-frequency patterns appear as false low-frequency structure: aliasing. Interpolation alone is not a general antialiasing filter. A one-pixel checkerboard reduced by two becomes a constant with our nearest tie rule; bilinear returns 0.5 for this particular pattern. That example does not prove bilinear prevents aliasing for all patterns. An explicit low-pass filter is needed before general downsampling.

In [ ]:
checker=(np.indices((8,8)).sum(0)%2)[...,None]
for method in ('nearest','bilinear'):
    print(method,resize_hwc(checker,(4,4),method)[:,:,0])
constant=np.full((3,4,2),7.)
np.testing.assert_allclose(resize_hwc(constant,(9,11)),7.)

## Real-data experiment and interpretation
Run `python scripts/image_foundations_experiments.py` from the repository root after the existing pinned pilot download. It verifies the training scene checksum, reads a fully valid 64×64 RGBNIR crop, averages nonoverlapping 2×2 cells to produce LR, then reconstructs with three methods. No test scene is used. Scaling is fixed stored values/10000, with physical calibration unverified.

On this crop bicubic had lower MSE in every band than bilinear and nearest. This is one example, not a universal ranking. The checkerboard illustrates aliasing; it is a synthetic diagnostic rather than a remote-sensing performance result. The next cell reads recorded evidence and shows the measured figure; it does not rerun the experiment.

In [ ]:
from IPython.display import display,Image
p=root/'reports/mp03-mp04-experiments.json'
if p.exists():
    print(json.loads(p.read_text())['mp03'])
    display(Image(filename=str(root/'reports/mp03-resizing.png')))
else: print('Real-data experiment not run in this checkout.')

## Practice
1. Calculate the bilinear value at the center of [[0,10],[20,30]]. Define all weights.
2. Resize a constant multiband image to an odd shape. Explain why the constant must be preserved.
3. Shift the checkerboard by one input pixel and repeat nearest downsampling. Predict the new constant.
4. Explain why a smooth resized image is not proof of recovered detail.

## Solution guide — read after attempting
1. a=b=0.5; each weight is 0.25, so the answer is 15.
2. The four weights sum to one independently in every band, including replicated boundaries.
3. The chosen checkerboard parity reverses, so constant zero becomes constant one. Sampling phase matters.
4. Interpolation estimates between samples; several different HR signals can share the same LR samples. Smoothness is not independent evidence of correctness.

In [ ]:
square=np.array([[0.,10.],[20.,30.]])[...,None]
assert resize_hwc(square,(1,1))[0,0,0]==15
np.testing.assert_array_equal(resize_hwc(1-checker,(4,4),'nearest'),1-resize_hwc(checker,(4,4),'nearest'))
print('Worked solution invariants passed; learner answers remain unassessed.')

## Reuse, execution and limitations
Implementation: `src/rssr/image_foundations.py`; checks: `python -m pytest tests/test_image_foundations.py`. See [setup](../../docs/setup.md) for local Jupyter and Colab clone/install instructions. Run from notebooks/ or the repository root. Scientific georeferencing is not handled by this NumPy utility; use the separate GeoTIFF pipeline for CRS/transform-preserving export.

[Authoritative brief](README.md), [dataset card](../../docs/dataset-cards.md), [measured record](../../reports/mp03-mp04-experiments.json). The existing PyTorch comparison is an implementation check; it is not proof of physical reconstruction accuracy.